# Dự báo vi khí hậu nhà màng — huấn luyện trên nhiều file CSV (Kaggle)

Notebook này đọc **nhiều file CSV** cảm biến (nhiệt độ, độ ẩm, ánh sáng) từ một
Kaggle Dataset, tự động **loại bỏ các khoảng thời gian bị lỗi**, rồi huấn luyện
mô hình dự báo trước 30 phút.

## Cách dùng trên Kaggle (làm 1 lần)

1. Vào **kaggle.com → Datasets → New Dataset**, kéo thả 5 file CSV của bạn vào,
   đặt tên dataset (ví dụ `greenhouse-sensor-data`) rồi bấm **Create**.
2. Vào **Code → New Notebook**, xoá hết cell mặc định, dán/tải toàn bộ các cell
   của notebook này vào.
3. Bên phải màn hình, mục **Input**, bấm **Add Input → Datasets**, chọn dataset
   vừa tạo ở bước 1.
4. Bấm **Run All** (hoặc chạy lần lượt từng cell theo thứ tự bên dưới).

Mỗi file CSV cần có tối thiểu 4 cột: `timestamp, temperature, humidity, light`.
Tên file và số lượng file không quan trọng — notebook tự tìm mọi `.csv` trong
`/kaggle/input/`.


## Bước 1 — Cài đặt & cấu hình

Các thư viện dưới đây đều có sẵn trên Kaggle, không cần cài thêm.
Chỉnh các tham số trong `CONFIG` nếu muốn đổi chu kỳ resample, thời gian dự báo, v.v.


In [ ]:
import glob
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)

CONFIG = dict(
    input_dir="/kaggle/input",       # Kaggle tự mount dataset vào đây
    output_dir="/kaggle/working",    # nơi lưu model, notebook tự tải file từ đây
    required_cols={"timestamp", "temperature", "humidity", "light"},

    step="5min",              # chu kỳ resample. Thiết bị thật (ESP32) phải dùng đúng chu kỳ này
    horizon_steps=6,          # 6 x 5 phút = dự báo trước 30 phút
    lags=(1, 2, 3, 6, 12),    # đặc trưng "độ chênh so với N bước trước"

    # ---- các ngưỡng dùng để LOẠI BỎ khoảng thời gian bị lỗi ----
    gap_break_min=10,         # mất dữ liệu quá 10 phút -> cắt thành đoạn mới, không nội suy
    interpolate_limit=3,      # chỉ nội suy khoảng trống <= 3 bước (15 phút)
    min_segment_hours=3,      # đoạn liên tục ngắn hơn 3 giờ -> bỏ, không đủ để tạo đặc trưng
    temp_range=(-10, 70),     # ngoài khoảng này -> lỗi cảm biến nhiệt độ
    hum_range=(0, 100),       # ngoài khoảng này -> lỗi cảm biến độ ẩm
    max_temp_jump_per_30s=3.0,   # nhiệt độ đổi > 3°C trong 1 lần đọc (30s) -> glitch tức thời
    max_hum_jump_per_30s=15.0,   # độ ẩm đổi > 15%RH trong 1 lần đọc (30s) -> glitch tức thời
    stuck_min_repeats=40,     # 1 giá trị lặp lại y hệt >= 40 lần liên tiếp -> nghi cảm biến bị kẹt

    light_saturation=54612.5,  # BH1750 bão hòa ở 65535/1.2 lux
    targets=["temperature", "humidity", "log_light"],
)
print("Đã nạp cấu hình.")


## Bước 2 — Tìm tất cả file CSV trong dataset đã gắn vào notebook


In [ ]:
csv_files = sorted(glob.glob(f"{CONFIG['input_dir']}/**/*.csv", recursive=True))
print(f"Tìm thấy {len(csv_files)} file CSV:")
for f in csv_files:
    print(" ", f)

if len(csv_files) == 0:
    raise SystemExit(
        "Không tìm thấy file CSV nào trong /kaggle/input. "
        "Kiểm tra lại đã 'Add Input' đúng dataset chưa (xem hướng dẫn ở đầu notebook)."
    )


## Bước 3 — Đọc từng file & loại bỏ khoảng thời gian bị lỗi

Với mỗi file, thứ tự xử lý là:

1. Ép kiểu timestamp, bỏ dòng không đọc được thời gian, sắp xếp theo thời gian, bỏ trùng.
2. **Giá trị vật lý không hợp lệ** (độ ẩm ngoài 0–100%, nhiệt độ ngoài khoảng hợp lý) → đánh dấu lỗi.
3. **Nhảy đột ngột bất thường** giữa 2 lần đọc liên tiếp (30s) vượt quá tốc độ thay đổi vật lý
   có thể có → đánh dấu lỗi (thường do nhiễu điện, dây lỏng).
4. **Cảm biến nhiệt độ bị kẹt** — nhiệt độ lặp lại y hệt quá lâu (thực tế luôn dao động nhỏ,
   đứng yên tuyệt đối là dấu hiệu cảm biến treo) → đánh dấu lỗi. Không áp dụng kiểu kiểm tra
   này cho độ ẩm/ánh sáng vì độ ẩm 100% (sương mù) và ánh sáng 0 (ban đêm) có thể đứng yên
   thật sự trong nhiều giờ — đó là hiện tượng bình thường, không phải lỗi.
5. Toàn bộ điểm bị đánh dấu lỗi ở trên → đổi thành NaN.
6. Cắt dữ liệu thành **các đoạn liên tục** tại những chỗ mất dữ liệu > `gap_break_min` phút
   (kể cả do bị lỗi ở bước 2–4 lẫn do mất kết nối LoRa thật sự). Notebook **không nối** hai đoạn
   cách nhau bởi lỗi/khoảng trống lớn, để không tạo ra đặc trưng "giả" bắc cầu qua chỗ hỏng.
7. Resample về chu kỳ cố định (`step`), chỉ nội suy các khoảng trống ngắn (≤ `interpolate_limit`
   bước), rồi bỏ các đoạn còn lại ngắn hơn `min_segment_hours`.


In [ ]:
def load_one_file(path, cfg):
    df = pd.read_csv(path)
    missing = cfg["required_cols"] - set(df.columns)
    if missing:
        print(f"  BỎ QUA {path}: thiếu cột {missing}")
        return None

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    n_raw = len(df)
    df = (df.dropna(subset=["timestamp"])
            .sort_values("timestamp")
            .drop_duplicates("timestamp")
            .set_index("timestamp"))

    err = pd.Series(False, index=df.index)

    # (2) giá trị vật lý không hợp lệ
    lo, hi = cfg["temp_range"]
    err |= (df["temperature"] < lo) | (df["temperature"] > hi) | df["temperature"].isna()
    lo, hi = cfg["hum_range"]
    err |= (df["humidity"] < lo) | (df["humidity"] > hi) | df["humidity"].isna()
    err |= (df["light"] < 0) | df["light"].isna()

    # (3) nhảy đột ngột bất thường giữa 2 lần đọc liên tiếp
    err |= df["temperature"].diff().abs() > cfg["max_temp_jump_per_30s"]
    err |= df["humidity"].diff().abs() > cfg["max_hum_jump_per_30s"]

    # (4) cảm biến bị kẹt: giá trị lặp lại y hệt quá lâu.
    # Chỉ áp dụng cho NHIỆT ĐỘ: nhiệt độ không khí thực tế luôn dao động dù rất nhỏ,
    # nên đứng yên tuyệt đối trong thời gian dài gần như chắc chắn là lỗi cảm biến/ADC treo.
    # KHÔNG áp dụng cho độ ẩm và ánh sáng, vì cả hai đều có thể đứng ở mức bão hoà thật:
    # độ ẩm = 100% (sương/mù kéo dài) và ánh sáng = 0 (ban đêm) đều là hiện tượng bình
    # thường trong nhà màng, không phải lỗi cảm biến — nếu lọc theo kiểu "lặp lại lâu"
    # sẽ xoá nhầm phần lớn dữ liệu ban đêm và sáng sớm.
    same_run = (df["temperature"] != df["temperature"].shift()).cumsum()
    run_len = same_run.map(same_run.value_counts())
    err |= run_len >= cfg["stuck_min_repeats"]

    n_err = int(err.sum())
    df.loc[err, ["temperature", "humidity", "light"]] = np.nan

    print(f"  {path.split('/')[-1]}: {n_raw} dòng -> {n_err} điểm bị đánh dấu lỗi "
          f"({n_err/max(n_raw,1)*100:.1f}%), khoảng {df.index.min()} .. {df.index.max()}")
    return df


def split_segments(df, source_name, cfg):
    valid = df["temperature"].notna() & df["humidity"].notna() & df["light"].notna()
    ts = df.index.to_series()
    # coi cả điểm lỗi (NaN) lẫn khoảng trống thời gian như nhau: đều là "không có dữ liệu tốt"
    gap_time = ts.diff() > pd.Timedelta(minutes=cfg["gap_break_min"])
    gap_err = (~valid) & (~valid.shift(fill_value=False))  # chuỗi lỗi liên tiếp
    seg_id = (gap_time | (~valid)).cumsum()

    segments = []
    for _, part in df[valid].groupby(seg_id[valid]):
        hours = (part.index[-1] - part.index[0]).total_seconds() / 3600
        if hours < cfg["min_segment_hours"]:
            continue
        seg = (part.resample(cfg["step"]).mean()
                    .interpolate(limit=cfg["interpolate_limit"])
                    .dropna())
        if len(seg) < 20:
            continue
        segments.append((source_name, seg))
    return segments


all_segments = []
for f in csv_files:
    df = load_one_file(f, CONFIG)
    if df is None:
        continue
    segs = split_segments(df, f.split("/")[-1], CONFIG)
    print(f"    -> giữ lại {len(segs)} đoạn liên tục sạch")
    all_segments.extend(segs)

print(f"\nTổng cộng {len(all_segments)} đoạn liên tục sạch từ {len(csv_files)} file")
if len(all_segments) < 2:
    raise SystemExit("Cần ít nhất 2 đoạn liên tục (tốt nhất ở nhiều ngày khác nhau) để đánh giá mô hình.")


## Bước 4 — Tạo đặc trưng (feature engineering)

Với mỗi đoạn liên tục sạch, notebook tạo:
- Giá trị hiện tại của 3 biến (ánh sáng lấy `log1p` vì dải giá trị rất rộng, từ 0 đến hơn 50.000 lux).
- Độ chênh so với 5 / 10 / 15 / 30 / 60 phút trước (xu hướng tăng/giảm).
- Trung bình & độ lệch chuẩn trong 30 phút gần nhất.
- Cờ báo cảm biến ánh sáng đang bão hòa.
- Giờ trong ngày dạng `sin`/`cos` (để mô hình biết chu kỳ ngày–đêm).

Nhãn cần dự báo là **mức thay đổi** sau `horizon_steps` bước (không phải giá trị tuyệt đối) —
cây quyết định không ngoại suy tốt ngoài dải đã học, nên dự báo độ lệch ổn định hơn nhiều.

**Quan trọng:** đặc trưng/nhãn được tính riêng cho từng đoạn liên tục, không bao giờ tính
bắc qua ranh giới giữa hai đoạn (dù là ranh giới do lỗi hay do khác file), để tránh sinh ra
mẫu huấn luyện giả từ chỗ dữ liệu bị đứt.


In [ ]:
def to_base(seg, cfg):
    return pd.DataFrame({
        "temperature": seg["temperature"],
        "humidity": seg["humidity"],
        "log_light": np.log1p(seg["light"]),
    })


def make_features(seg, cfg):
    base = to_base(seg, cfg)
    sat = (seg["light"] >= cfg["light_saturation"] * 0.99).astype(float)

    X = pd.DataFrame(index=seg.index)
    for name in cfg["targets"]:
        s = base[name]
        X[name] = s
        for k in cfg["lags"]:
            X[f"{name}_d{k}"] = s - s.shift(k)
        X[f"{name}_mean6"] = s.rolling(6).mean()
        X[f"{name}_std6"] = s.rolling(6).std()
    X["light_sat6"] = sat.rolling(6).mean()

    h = seg.index.hour + seg.index.minute / 60
    X["hour_sin"] = np.sin(2 * np.pi * h / 24)
    X["hour_cos"] = np.cos(2 * np.pi * h / 24)
    return X


def build(seg, cfg):
    X = make_features(seg, cfg)
    base = to_base(seg, cfg)
    Y = pd.DataFrame({t: base[t].shift(-cfg["horizon_steps"]) - base[t] for t in cfg["targets"]})
    ok = X.notna().all(axis=1) & Y.notna().all(axis=1)
    return X[ok], Y[ok]


parts, srcs = [], []
for src, seg in all_segments:
    X, Y = build(seg, CONFIG)
    if len(X) == 0:
        continue
    parts.append((X, Y))
    srcs.append(pd.Series(src, index=X.index))

X_all = pd.concat([p[0] for p in parts])
Y_all = pd.concat([p[1] for p in parts])
src_all = pd.concat(srcs)

print(f"Tổng {len(X_all)} mẫu huấn luyện, {X_all.shape[1]} đặc trưng, "
      f"dự báo trước {CONFIG['horizon_steps'] * int(CONFIG['step'].replace('min',''))} phút")
print("\nSố mẫu theo từng file nguồn:")
print(src_all.value_counts().to_string())


## Bước 5 — Đánh giá bằng leave-one-day-out

Notebook lần lượt giữ lại **từng ngày** làm tập kiểm tra, huấn luyện trên tất cả các ngày
còn lại (từ mọi file), rồi so sánh sai số với **baseline "giữ nguyên giá trị hiện tại"**.
Có vùng đệm (`purge`) 1 giờ quanh ngày kiểm tra để nhãn của tập train không chồng lấn sang
tập test. Đây là cách đánh giá trung thực nhất với dữ liệu dạng chuỗi thời gian.


In [ ]:
PURGE = pd.Timedelta(hours=1)
day = X_all.index.normalize()

rows = []
for d in sorted(day.unique()):
    te = day == d
    if te.sum() < 50:          # ngày có quá ít mẫu thì chỉ dùng để train, không test
        continue
    lo, hi = X_all.index[te].min() - PURGE, X_all.index[te].max() + PURGE
    tr = (X_all.index < lo) | (X_all.index > hi)

    r = {"ngày": str(d.date()), "n_test": int(te.sum())}
    for t in CONFIG["targets"]:
        m = ExtraTreesRegressor(n_estimators=40, max_depth=5, min_samples_leaf=15,
                                 n_jobs=-1, random_state=0).fit(X_all[tr], Y_all.loc[tr, t])
        y = Y_all.loc[te, t].values
        p = m.predict(X_all[te])
        r[f"{t}_baseline"] = float(np.mean(np.abs(y)))
        r[f"{t}_model"] = float(np.mean(np.abs(y - p)))
    rows.append(r)

daily = pd.DataFrame(rows).set_index("ngày")
print("Sai số MAE theo từng ngày (nhiệt độ °C | độ ẩm %RH | log_light):")
print(daily.round(3).to_string())

summary = pd.DataFrame({
    t: {
        "baseline": daily[f"{t}_baseline"].mean(),
        "mô hình": daily[f"{t}_model"].mean(),
    } for t in CONFIG["targets"]
}).T
summary["cải thiện %"] = (1 - summary["mô hình"] / summary["baseline"]) * 100
print("\nTổng hợp:")
print(summary.round(3).to_string())
print("\n'cải thiện %' > 0 nghĩa là mô hình thắng baseline. "
      "Nên đạt ổn định 10-15%+ trên nhiều ngày trước khi triển khai điều khiển thiết bị.")


## Bước 6 — Huấn luyện mô hình cuối cùng trên toàn bộ dữ liệu & lưu lại

Sau khi đã xem kết quả đánh giá ở Bước 5 và thấy chấp nhận được, huấn luyện lại trên
**toàn bộ** dữ liệu sạch (không giữ lại tập test) để lấy mô hình dùng thật.

File lưu vào `/kaggle/working/` — vào tab **Output** bên phải notebook để tải về:
- `model_temperature.joblib`, `model_humidity.joblib`, `model_log_light.joblib`
- `features.json` — danh sách đặc trưng đúng thứ tự, để bên chạy suy luận (master) tính đúng.


In [ ]:
import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)

total_nodes = 0
for t in CONFIG["targets"]:
    final_model = ExtraTreesRegressor(n_estimators=40, max_depth=5, min_samples_leaf=15,
                                       n_jobs=-1, random_state=0).fit(X_all, Y_all[t])
    total_nodes += sum(e.tree_.node_count for e in final_model.estimators_)
    path = f"{CONFIG['output_dir']}/model_{t}.joblib"
    joblib.dump(final_model, path)
    print("Đã lưu", path)

with open(f"{CONFIG['output_dir']}/features.json", "w") as f:
    json.dump({
        "features": list(X_all.columns),
        "step": CONFIG["step"],
        "horizon_steps": CONFIG["horizon_steps"],
        "targets": CONFIG["targets"],
    }, f, indent=2, ensure_ascii=False)
print("Đã lưu features.json")
print(f"\nTổng số node của 3 mô hình: {total_nodes} "
      f"(~{total_nodes * 12 / 1024:.0f} KB nếu sau này xuất sang C cho ESP32)")


## Bước 7 — Biểu đồ kiểm tra trực quan

So sánh giá trị thực tế, dự báo của mô hình, và baseline trên từng đoạn liên tục sạch,
để mắt thường kiểm tra mô hình có bám sát xu hướng thật hay không (không chỉ nhìn số MAE).


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=False)
final_models = {t: joblib.load(f"{CONFIG['output_dir']}/model_{t}.joblib") for t in CONFIG["targets"]}

offset = pd.Timedelta(0)
for src, seg in all_segments:
    X, Y = build(seg, CONFIG)
    if len(X) == 0:
        continue
    idx = X.index + offset   # dịch trục thời gian để các đoạn không đè lên nhau trên biểu đồ
    for ax, t in zip(axes, CONFIG["targets"]):
        pred = final_models[t].predict(X)
        ax.plot(idx, X[t] + Y[t], color="tab:blue", lw=1.2,
                label="Thực tế" if src == all_segments[0][0] else None)
        ax.plot(idx, X[t] + pred, color="tab:orange", lw=1.0,
                label="Mô hình" if src == all_segments[0][0] else None)
    offset += (X.index.max() - X.index.min()) + pd.Timedelta(hours=6)

for ax, t in zip(axes, CONFIG["targets"]):
    ax.set_ylabel(t)
    ax.legend(loc="upper right", fontsize=8)
axes[0].set_title(f"Dự báo trước {CONFIG['horizon_steps']*5} phút — các đoạn dữ liệu sạch nối tiếp nhau (có khoảng trắng phân cách)")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/forecast_plot.png", dpi=110)
plt.show()
print("Đã lưu forecast_plot.png vào", CONFIG["output_dir"])


## Bước 8 (tuỳ chọn) — Chuẩn bị xuất mô hình cho ESP32

Notebook này lưu mô hình dạng `.joblib` để dùng ở **master (Raspberry Pi / server)**.
Nếu sau này muốn nhúng thẳng vào ESP32, cài thêm `emlearn` (không có sẵn trên Kaggle, chạy
ở máy cá nhân) rồi chuyển từng cây trong `ExtraTreesRegressor` sang mã C:

```python
# chạy ở máy cá nhân, không phải trên Kaggle
import emlearn
model = joblib.load("model_temperature.joblib")
cmodel = emlearn.convert(model, method="inline")
cmodel.save(file="forecast_temperature.h", name="forecast_temperature")
```

Xem lại số node ở Bước 6 để ước lượng dung lượng bộ nhớ cần trên ESP32.
